## Librerias

In [21]:
import geopandas as gpd
import pandas as pd
import unicodedata

## Cargando ambos datos

In [22]:
df_organizacion_territorial = gpd.read_file(
    "ORGANIZACIÓN TERRITORIAL PARROQUIAL 03.02.2026/ORGANIZACION_TERRITORIAL_PARROQUIAL.shp"
)
df_organizacion_territorial = df_organizacion_territorial[["DPA_PARROQ", "DPA_DESPAR","DPA_DESCAN",	"DPA_DESPRO"]]
print(df_organizacion_territorial.shape)
df_organizacion_territorial.head(3)

(1050, 4)


,DPA_PARROQ,DPA_DESPAR,DPA_DESCAN,DPA_DESPRO
0,010150,CUENCA,CUENCA,AZUAY
1,010151,BAÑOS,CUENCA,AZUAY
2,010152,CUMBE,CUENCA,AZUAY


In [23]:
df_densidad_poblacional= pd.read_csv("densidad_POBLACIONAL_BASE.csv", usecols= ["Provincia", "Cantón", "Parroquia", "Densidad Poblacional"])
#Renombrando columnas
df_densidad_poblacional = df_densidad_poblacional.rename(columns={
    "Provincia": "DPA_DESPRO",
    "Parroquia": "DPA_DESPAR",
    "Cantón": "DPA_DESCAN"
})
print(df_densidad_poblacional.shape)
df_densidad_poblacional.head(3)

(1042, 4)


,DPA_DESPRO,DPA_DESCAN,DPA_DESPAR,Densidad Poblacional
0,AZUAY,CUENCA,CUENCA,5.044
1,AZUAY,CUENCA,BAÑOS,87.000
2,AZUAY,CUENCA,CUMBE,86.000


## Normalizando columnas para unirlos mediante columnas

In [24]:
def normalizar(texto):

    if pd.isna(texto):
        return texto

    #Eliminar espacios al inicio y al final, y coloca en mayusculas
    texto = str(texto).strip().upper()

    # Eliminar tildes y diéresis
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

    # Eliminar espacios dobles
    texto = ' '.join(texto.split())

    return texto

df_organizacion_territorial["DPA_DESPAR"] = df_organizacion_territorial["DPA_DESPAR"].apply(normalizar)
df_densidad_poblacional["DPA_DESPAR"] = df_densidad_poblacional["DPA_DESPAR"].apply(normalizar)

df_organizacion_territorial["DPA_DESPRO"] = df_organizacion_territorial["DPA_DESPRO"].apply(normalizar)
df_densidad_poblacional["DPA_DESPRO"] = df_densidad_poblacional["DPA_DESPRO"].apply(normalizar)

df_organizacion_territorial["DPA_DESCAN"] = df_organizacion_territorial["DPA_DESCAN"].apply(normalizar)
df_densidad_poblacional["DPA_DESCAN"] = df_densidad_poblacional["DPA_DESCAN"].apply(normalizar)

## Viendo filas faltantes

In [25]:
df_densidad_poblacional.columns

Index(['DPA_DESPRO', 'DPA_DESCAN', 'DPA_DESPAR', 'Densidad Poblacional'], dtype='object')

In [26]:
columnas = ['DPA_DESPRO', 'DPA_DESCAN', 'DPA_DESPAR']

df_faltantes = df_organizacion_territorial[
    ~df_organizacion_territorial.set_index(columnas).index.isin(df_densidad_poblacional.set_index(columnas).index)
]
print(len(df_faltantes))
df_faltantes


10


,DPA_PARROQ,DPA_DESPAR,DPA_DESCAN,DPA_DESPRO
447,100153,LA CAROLINA,IBARRA,IMBABURA
660,131251,SOSOTE,ROCAFUERTE,MANABI
696,140190,"ZONA EN ESTUDIO ""SINAI-CUCHAENTZA""",MORONA,MORONA SANTIAGO
746,141350,SEVILLA DON BOSCO,SEVILLA DON BOSCO,MORONA SANTIAGO
786,160167,SHUAR PASTAZA,PASTAZA,PASTAZA
835,170257,JUAN MONTALVO,CAYAMBE,PICHINCHA
979,210456,LA MAGDALENA,SHUSHUFINDI,SUCUMBIOS
980,210457,LA PRIMAVERA,SHUSHUFINDI,SUCUMBIOS
1048,900651,ZONA EN ESTUDIO: JUVAL (CANAR-CHIMBORAZO),ZONA EN ESTUDIO: JUVAL (CANAR-CHIMBORAZO),ZONA EN ESTUDIO: JUVAL (CANAR-CHIMBORAZO)
1049,ISLA,ISLA,ISLA,ISLA


Se ignora por que son poco datos y no hay densidad poblacional para aquellas parroquias

## Uniendo mediante columnas de "DPA_DESPRO", "DPA_DESCAN","DPA_DESPAR" y agregando geometria a densidad poblacional

In [27]:
df_densidad_poblacional = df_densidad_poblacional.merge(
    df_organizacion_territorial[
        ["DPA_DESPRO", "DPA_DESCAN","DPA_DESPAR", "DPA_PARROQ"]
    ],
    on=["DPA_DESPRO", "DPA_DESCAN","DPA_DESPAR"],
    how="left"
)
df_densidad_poblacional.head(3)

,DPA_DESPRO,DPA_DESCAN,DPA_DESPAR,Densidad Poblacional,DPA_PARROQ
0,AZUAY,CUENCA,CUENCA,5.044,010150
1,AZUAY,CUENCA,BANOS,87.000,010151
2,AZUAY,CUENCA,CUMBE,86.000,010152


## Eliminando "DPA_DESPRO", "DPA_DESCAN", "DPA_DESPAR" de densidad poblacional para colocar las de organizacion_territorial mediante geometria

In [28]:
df_densidad_poblacional = df_densidad_poblacional.drop(columns= ["DPA_DESPRO", "DPA_DESCAN", "DPA_DESPAR"])

In [29]:
df_densidad_poblacional = df_densidad_poblacional.set_index('DPA_PARROQ') #Colocamos de indice el codigo de parroquia
df_densidad_poblacional.head(3)

,Densidad Poblacional
DPA_PARROQ,
010150,5.044
010151,87.000
010152,86.000


In [30]:
df_densidad_poblacional.to_csv("densidad_poblacional.csv")#Transformamos a csv